In [ ]:
## MobileNetV2 Model

In [ ]:
## MobileNetV2 Model (finetuned)

In [ ]:
import tensorflow as tf
from keras.applications import MobileNetV2

conv_base = MobileNetV2(
    weights='imagenet',
    include_top=False,
    input_shape=(224, 224, 3)
)

# freeze all training
conv_base.trainable = False

In [ ]:
from keras.optimizers import Adam
inputs = layers.Input(shape=(224, 224,3))
x = preprocess_input(inputs)
x = conv_base(x, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dense(256, activation='relu')(x)
x = layers.Dropout(0.5)(x)
outputs = layers.Dense(num_classes, activation='softmax')(x)
model = keras.Model(inputs, outputs)

model.summary()

In [ ]:
model.compile(optimizer=Adam(learning_rate=1e-3), loss='sparse_categorical_crossentropy', metrics=['accuracy'])

In [ ]:
callbacks = [
    keras.callbacks.ModelCheckpoint(
        filepath="feature_extraction.keras",
        save_best_only=True,
        monitor="val_loss",
    )
]

mobilenet_history = model.fit(
    train_dataset,
    batch_size=batch_size_n,
    epochs=5,
    validation_data=val_dataset,
    callbacks=callbacks,
)

In [ ]:
conv_base.trainable = True

for layer in conv_base.layers[:-4]:
    layer.trainable = False
    
for layer in conv_base.layers[-4:]:
    layer.trainable = True

model.compile(optimizer=Adam(learning_rate=1e-5), loss='sparse_categorical_crossentropy', metrics=['accuracy'])    

In [ ]:
from keras.callbacks import EarlyStopping

finetuned_history = model.fit(
    train_dataset,
    epochs=64,
    validation_data=val_dataset,
    callbacks=[
        EarlyStopping(patience=3, restore_best_weights=True)
    ]
)

In [ ]:
acc = finetuned_history.history["accuracy"]
val_acc = finetuned_history.history["val_accuracy"]
loss = finetuned_history.history["loss"]
val_loss = finetuned_history.history["val_loss"]
epochs = range(1, len(acc) + 1)
plt.plot(epochs, acc, "r--", label="Training accuracy")
plt.plot(epochs, val_acc, "b", label="Validation accuracy")
plt.title("Training and validation accuracy")
plt.legend()
plt.figure()
plt.plot(epochs, loss, "r--", label="Training loss")
plt.plot(epochs, val_loss, "b", label="Validation loss")
plt.title("Training and validation loss")
plt.legend()
plt.show()

In [ ]:
test_loss, test_acc = model.evaluate(test_dataset)
val_loss, val_acc = model.evaluate(val_dataset)
print("Test accuracy:", test_acc)
print("Validation accuracy:", val_acc)

In [ ]:
## EfficientNet Model